# Deploy Lakebase

Create the Lakebase PostgreSQL instance, schema, tables, UC view,
and Reverse ETL synced table for sub-10 ms DICOMweb lookups.

In [0]:
import os
_nb_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_repo_root = os.path.normpath(os.path.join("/Workspace" + os.path.dirname(_nb_ctx.notebookPath().get()), ".."))

%pip install --quiet -r {_repo_root}/requirements.txt
dbutils.library.restartPython()

In [0]:
%run ./config/proxy_prep

In [0]:
sql_warehouse_id, table, volume = init_widgets(show_volume=True)
init_env()

In [ ]:
lakebase_instance_name = dbutils.widgets.get("lakebase_instance_name")

# Create Lakebase Instance, Schema, and Tables

In [0]:
import os
import dbx
from dbx.pixels.lakebase import LakebaseUtils, parse_uc_table_name

lb_utils = LakebaseUtils(instance_name=lakebase_instance_name, uc_table_name=table, create_instance=True, min_cu=0.5, max_cu=2.0)

path = os.path.dirname(dbx.pixels.__file__)
sql_base_path = f"{path}/resources/sql/lakebase"

_lb_schema = lb_utils.schema

for sql_file in ["CREATE_LAKEBASE_SCHEMA.sql", "CREATE_LAKEBASE_DICOM_FRAMES.sql", "CREATE_LAKEBASE_METRICS.sql"]:
    file_path = os.path.join(sql_base_path, sql_file)
    with open(file_path, "r") as file:
        lb_utils.execute_query(file.read().format(schema_name=_lb_schema))

# Create the UC view used by Reverse ETL to sync instance_paths into Lakebase
_uc_catalog, _uc_schema, _uc_table = parse_uc_table_name(table)
with open(os.path.join(sql_base_path, "CREATE_INSTANCE_PATHS_VIEW.sql"), "r") as file:
    spark.sql(file.read().format(catalog=_uc_catalog, schema=_uc_schema, table=_uc_table))
print(f"✓ Created UC view {_uc_catalog}.{_uc_schema}.instance_paths_vw for Reverse ETL Sync")

# Create Reverse ETL Synced Table

Sync the `instance_paths_vw` view from Unity Catalog into Lakebase using
Reverse ETL.  This creates:

1. A **read-only synced table** in UC (`catalog.schema.instance_paths`)
2. A **Postgres table** in Lakebase (`schema.instance_paths`)

The sync pipeline keeps the Lakebase table updated so the
DICOMweb app can resolve SOP Instance UIDs → file paths in sub-10 ms
without querying the SQL warehouse.

In [0]:
import json, time

_synced_table_name = f"{_uc_catalog}.{_uc_schema}.instance_paths"

# Get project and branch UIDs from the Lakebase postgres project
_project = lb_utils.workspace_client.postgres.get_project(lb_utils.project_resource_name)
_branch = lb_utils.workspace_client.postgres.get_branch(lb_utils.branch_resource_name)

# Check if synced table already exists in Unity Catalog
_needs_create = True
try:
    _tbl = w.tables.get(_synced_table_name)
    if _tbl.table_type and _tbl.table_type.value == "FOREIGN":
        print(f"✓ Synced table '{_synced_table_name}' already exists (pipeline_id={_tbl.pipeline_id})")
        _needs_create = False
    else:
        print(f"⚠ Table '{_synced_table_name}' exists but is not a synced table (type={_tbl.table_type})")
        _needs_create = False
except Exception:
    pass  # Table doesn't exist in UC — need to create it

if _needs_create:
    # Drop any stale Postgres table left from a previous failed sync
    try:
        lb_utils.execute_query(f"DROP TABLE IF EXISTS {_lb_schema}.instance_paths")
    except Exception:
        pass  # Table may not exist — that's fine

    _payload = {
        "name": _synced_table_name,
        "database_project_id": _project.uid,
        "database_branch_id": _branch.uid,
        "logical_database_name": _uc_catalog,
        "spec": {
            "source_table_full_name": f"{_uc_catalog}.{_uc_schema}.instance_paths_vw",
            "primary_key_columns": ["local_path"],
            "scheduling_policy": "SNAPSHOT",
        },
    }

    try:
        _result = w.api_client.do("POST", "/api/2.0/database/synced_tables", body=_payload)
        _pipeline_id = _result.get("data_synchronization_status", {}).get("pipeline_id", "unknown")
        print(f"✓ Synced table created — pipeline {_pipeline_id} running")

        # Poll until online or failed (max ~2 min)
        for _i in range(12):
            time.sleep(10)
            _status_resp = w.api_client.do("GET", f"/api/2.0/database/synced_tables/{_synced_table_name}")
            _state = _status_resp.get("data_synchronization_status", {}).get("detailed_state", "")
            if "ONLINE" in _state:
                print(f"✓ Synced table is online")
                break
            elif "FAIL" in _state:
                _msg = _status_resp.get("data_synchronization_status", {}).get("message", "")
                print(f"✗ Synced table failed: {_msg}")
                break
    except Exception as _e:
        print(f"✗ Failed to create synced table: {_e}")

In [0]:
print("✅ Lakebase deployment complete")